# Introduction

This is a Notebook used to collect data from BBC News RSS Feeds.

The Notebook is run with a certain frequency to collect new data.
Existing data (read from database) is merged (removing duplicates) with the new data.
Then the resulting updated data is saved as new version of the database.


We also exemplify here how to use Neptune.ai with Kaggle

# Install and import packages

In [ ]:
!pip3 install requests_html

In [ ]:
!pip3 install neptune-client==1.2.0

In [ ]:
import requests
import pandas as pd
from requests_html import HTML
from requests_html import HTMLSession
from bs4 import BeautifulSoup
import neptune.new as neptune
from kaggle_secrets import UserSecretsClient

# RSS Feed Parsing Functions

In [ ]:
def get_html_source(url):
    """
        Return the source code for the provided URL. 
        source: https://practicaldatascience.co.uk/data-science/how-to-read-an-rss-feed-in-python
    Args: 
        url (string): URL of the page to scrape.

    Returns:
        response (object): HTTP response object from requests_html. 
    """

    try:
        session = HTMLSession()
        response = session.get(url)
        return response

    except requests.exceptions.RequestException as ex:
        print(ex)

In [ ]:
def get_rss_feed(url):
    """
       Return a Pandas dataframe containing the RSS feed contents.
       Source: https://practicaldatascience.co.uk/data-science/how-to-read-an-rss-feed-in-python
       Modified to use BeautifulSoup (b4)
       
    Args: 
        url (string): URL of the RSS feed to read.

    Returns:
        df (dataframe): Pandas dataframe containing the RSS feed contents.
    """
    
    response = get_html_source(url)
    
    df = pd.DataFrame(columns = ['title', 'pubDate', 'guid', 'link', 'description'])

    with response as r:   
        # we use BeautifulSoup with `lxml-xml` type to parse the rss feed
        soup = BeautifulSoup(r.text , 'lxml-xml')
        items = soup.find_all('item')

        for item in items:   
            try:
                title = item.find('title').text
                pubDate = item.find('pubDate').text
                guid = item.find('guid').text
                link = item.find('link').text
                description = item.find('description').text

                row = {'title': title, 'pubDate': pubDate, 'guid': guid, 'link': link, 'description': description}
                df = df.append(row, ignore_index=True)
            except Exception as ex:
                print(ex)
                continue
    return df

In [ ]:
user_secrets = UserSecretsClient()
neptune_api_token = user_secrets.get_secret("neptune_api")
run = None
try:
    run = neptune.init(
        project="preda/BBCNews",
        api_token=neptune_api_token,
    )  # your credentials
except Exception as ex:
    print(ex)

# Read BBC News RSS Feeds

Initialize the RSS Feed url.

In [ ]:
url = "http://feeds.bbci.co.uk/news/rss.xml"

Get the RSS Feed.

In [ ]:
data_df = get_rss_feed(url)
if run:
    run["new_data_rows"] = data_df.shape[0]
    run["new_data_columns"] = data_df.shape[1]
print(f"New data collected: {data_df.shape[0]}")
data_df.head()

# Load data from database and concatenate old and new data

Load the data from database.

In [ ]:
old_data_df = pd.read_csv("/kaggle/input/bbc-news/bbc_news.csv")
if run:
    run["old_data_rows"] = old_data_df.shape[0]
    run["old_data_columns"] = old_data_df.shape[1]
print(f"Old data: {old_data_df.shape[0]}")
old_data_df.head()

Let's look also to the dataset tail.

In [ ]:
old_data_df.tail()

Merge the newly parsed data with existing one.
Remove duplicates.

In [ ]:
new_data_df = pd.concat([old_data_df, data_df], axis=0)
print(f"Data after concatenation: {new_data_df.shape[0]}")
new_data_df = new_data_df.drop_duplicates()
if run:
    run["merged_data_rows"] = new_data_df.shape[0]
    run["merged_data_columns"] = new_data_df.shape[1]
print(f"Data after droping duplicates: {new_data_df.shape[0]}")
new_data_df.head()

Let's look also to new dataset tail.

In [ ]:
new_data_df.tail()

# Save merged data

After merging the data, save it (this will populate the next version of dataset).

In [ ]:
new_data_df.to_csv("bbc_news.csv", index=False)

# Stop Neptune.ai session

In [ ]:
if run:
    run.stop()